# Score a BENDL ensemble

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

<div style="text-align: center;"><a class="sd-sphinx-override sd-btn sd-text-wrap sd-btn-primary reference external" href="https://www.dropbox.com/scl/fo/s22x9phl0hldiakn8nbuz/ABKfxHBaak5ra3eBGkNFWMM?rlkey=igpo7qi07oz5tfgjki317o79t&amp;st=gcxkicnc&amp;dl=1">Download tutorial data</a></div>

Top-level lowercase functions evaluate one plan directly. Capitalized metric descriptions can
instead be registered with `PlanEvaluator`, which prepares graph and geometry resources once and
reuses the Rust scoring engine for one plan, selected plans, or an encoded ensemble. Functions in
`scoring.formulas` operate on arrays that have already been aggregated by district.

This guide uses one committed
Colorado VTD BENDL fixture (`data/co_vtd_scoring_10000.bendl`). The bundle contains:

- a 10,000-step, TwoDelta-encoded ReCom-B chain;
- its 3,158-node dual graph and run metadata;
- a GeoParquet asset with projected VTD geometry, population, and election columns; and
- fixture provenance, including the two small connectivity repairs made to the VTD seed.

The chain uses district-pairs MST, a 5% population tolerance, one thread, and batch size one.
The GeoParquet geometry is in NAD83 / Conus Albers (EPSG:5070). These are reproducible tutorial
fixtures, not a replacement for current official Colorado data.

In [ ]:
import tempfile
from io import BytesIO
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from binary_ensemble import BendlDecoder

import gerrytools.scoring as gs

## Open the BENDL resources

`BendlDecoder` verifies the checksums, reads the embedded graph, and exposes arbitrary assets.
GeoPandas can read the GeoParquet directly from its bytes. Assignment positions in the BENDL
stream follow the embedded graph's node order, and the `node` column records the same order
explicitly.

In [ ]:
data_dir = Path("data")
bundle_path = data_dir / "co_vtd_scoring_10000.bendl"

bundle = BendlDecoder(bundle_path)
bundle.verify()
graph = bundle.read_graph()
vtds = gpd.read_parquet(BytesIO(bundle.read_asset_bytes("co_vtds_2020.parquet")))
fixture_metadata = bundle.read_json_asset("fixture_metadata.json")

In [ ]:
pd.Series(
    {
        "VTDs": len(vtds),
        "graph edges": graph.number_of_edges(),
        "districts": vtds["assignment"].nunique(),
        "chain samples": bundle.count_samples(),
        "CRS": vtds.crs.to_string(),
        "assets": ", ".join(bundle.asset_names()),
    },
    name="Colorado scoring fixture",
)

## Configure an evaluator

The evaluator records borrowed graph and geometry sources when it is configured, then snapshots
only the requested resources during the first evaluation. Later evaluations reuse that immutable
snapshot and the scoring engine. Do not mutate either source after preparation; adding another
metric extends the snapshot only with resources that were not already prepared. The active
GeoDataFrame geometry column is reserved for geometry-backed metrics. This example registers
representative district, plan, region, election, and compactness metrics:

- `Tally` combines all requested numeric columns into one engine pass.
- `PolsbyPopper` demonstrates both graph measurements and geometry.
- `CutEdges` demonstrates unweighted counts and shared-perimeter weights.
- The three region statistics use counties as fixed regions.
- `Eguia` computes its fixed county benchmark during evaluator preparation.
- `TallyByRegion` produces a county-by-district table with named values.

Each metric owns its optional result `name`, so aliases work naturally with `add_metrics(...)`.
Compatible metrics still share engine state, but that implementation detail never changes how
results are accessed.

In [ ]:
evaluator = gs.PlanEvaluator(graph, geometry=vtds, node_column="node")
evaluator.add_metrics(
    gs.Tally(
        "total_pop_20",
        "total_vap_20",
        "bvap_20",
        "pres_16_dem",
        "pres_16_rep",
        "pres_20_dem",
        "pres_20_rep",
        "pres_24_dem",
        "pres_24_rep",
        name="district_totals",
    ),
    gs.PolsbyPopper(
        source="graph",
        perimeter="perimeter",
        name="polsby_popper_graph",
    ),
    gs.CutEdges(name="cut_edge_count"),
    gs.CutEdges("shared_perim", name="cut_edge_perimeter"),
    gs.RegionSplits("county", name="county_splits"),
    gs.RegionPieces("county", name="county_pieces"),
    gs.RegionParts("county", name="county_parts"),
    gs.Eguia(
        party_votes="pres_20_dem",
        opposition_votes="pres_20_rep",
        region="county",
        population="total_pop_20",
        name="eguia_2020",
    ),
    gs.TallyByRegion(
        "county",
        columns={
            "population": "total_pop_20",
            "democratic_votes": "pres_20_dem",
            "republican_votes": "pres_20_rep",
        },
        include_count=True,
        name="county_totals",
    ),
)

## Evaluate one assignment with `lookup`

`lookup(i)` returns one assignment vector without decoding earlier plans. For a one-off score,
call a lowercase function with a GeoDataFrame and either an assignment column or assignment
vector. The function constructs a temporary evaluator and returns a pandas object or scalar.
Geometry scores use the same pattern, for example `gs.polsby_popper(vtds, "assignment")`; use a
persistent evaluator when the statewide geometry will be reused.

In [ ]:
single_assignment = bundle.lookup(0)
direct_population = gs.tally(vtds, single_assignment, columns="total_pop_20")
direct_eguia = gs.eguia(
    vtds,
    "assignment",
    party_votes="pres_20_dem",
    opposition_votes="pres_20_rep",
    region="county",
    population="total_pop_20",
)
pd.DataFrame({"population": direct_population, "Eguia": direct_eguia})

In [ ]:
single = evaluator.evaluate(single_assignment)
single.metrics

In [ ]:
single["district_totals"]

In [ ]:
pd.Series(
    {name: single[name] for name in ("county_splits", "county_pieces", "county_parts")},
    name="county metrics",
)

Graph-backed Polsby-Popper uses the precomputed unit areas, perimeters, and shared boundary
lengths in the embedded graph. That representation supports efficient incremental updates over
the complete chain.

In [ ]:
single["polsby_popper_graph"]

## Evaluate selected assignments with `subsample_indices`

`subsample_indices` decodes only the requested zero-based sample indices. Materialize that small
selection before passing it to `evaluate_many`; `sample_ids` then assigns meaningful, unique
labels to the result rows rather than requiring the caller to relabel each table. Every
assignment in a batch must use the same district-label set.

In [ ]:
sample_indices = [0, 100, 1_000, 9_999]
selected_assignments = list(bundle.subsample_indices(sample_indices))
selected = evaluator.evaluate_many(
    selected_assignments,
    sample_ids=sample_indices,
)
selected["cut_edge_count"]

## Stream the complete 10,000-step chain

`evaluate_stream` accepts BEN, XBEN, and finalized BENDL input. It writes bounded,
Snappy-compressed Parquet batches into a temporary run directory and publishes the requested
directory only after every metric and the version-1 manifest finish successfully. The output
directory must not already exist.

`samples` counts expanded chain steps. `accepted` counts encoded frames written to each table.
When consecutive assignments repeat, the Parquet `repetitions` column preserves their multiplicity.

In [ ]:
run_root = Path(tempfile.mkdtemp(prefix="gerrytools-scoring-"))
run_dir = run_root / "scores"
run = evaluator.evaluate_stream(bundle_path, run_dir)
run.summary, run.metrics

In [ ]:
run.frames.head()

## Apply array formulas to streamed tallies

`EvaluationRun.read` reconstructs semantic pandas results without exposing physical Parquet
column names. With `expand_repetitions=True`, the sample index covers all 10,000 chain steps.
Eager reads warn at a predicted 2 GiB peak and raise `EvaluationMemoryError` at 8 GiB. For a
rejected read, process `run.iter_batches(...)` results individually instead of concatenating
them. Pass `allow_large=True` only when the machine deliberately has enough memory. Result
reads decode Parquet columns serially for a more predictable peak, which can reduce throughput
on wide metrics.
Functions in `scoring.formulas` accept arrays and preserve batch axes. The last axis is districts;
functions that combine elections expect elections on the penultimate axis.

In [ ]:
tallies = run.read("district_totals", expand_repetitions=True)
districts = tallies["total_pop_20"].columns

dem_2020 = tallies["pres_20_dem"].to_numpy()
rep_2020 = tallies["pres_20_rep"].to_numpy()
vote_shares_2020 = gs.formulas.district_vote_shares(dem_2020, rep_2020)
wins_2020 = gs.formulas.district_wins(dem_2020, rep_2020)

pd.concat(
    {
        "Democratic two-party share": pd.DataFrame(
            vote_shares_2020[:5],
            columns=districts,
        ),
        "Democratic win": pd.DataFrame(
            wins_2020[:5],
            columns=districts,
        ),
    },
    axis="columns",
).rename_axis(index="sample", columns=["quantity", "district"])

In [ ]:
partisan_scores_2020 = pd.DataFrame(
    {
        "seats": gs.formulas.seats(dem_2020, rep_2020),
        "overall_vote_share": gs.formulas.overall_vote_share(dem_2020, rep_2020),
        "efficiency_gap": gs.formulas.efficiency_gap(dem_2020, rep_2020),
        "simplified_efficiency_gap": gs.formulas.simplified_efficiency_gap(
            dem_2020,
            rep_2020,
        ),
        "mean_median": gs.formulas.mean_median(dem_2020, rep_2020),
        "partisan_bias_equal": gs.formulas.partisan_bias(
            dem_2020,
            rep_2020,
            turnout_model="equal",
        ),
        "partisan_bias_observed": gs.formulas.partisan_bias(
            dem_2020,
            rep_2020,
            turnout_model="observed",
        ),
        "partisan_gini_equal": gs.formulas.partisan_gini(
            dem_2020,
            rep_2020,
            turnout_model="equal",
        ),
        "partisan_gini_observed": gs.formulas.partisan_gini(
            dem_2020,
            rep_2020,
            turnout_model="observed",
        ),
    }
)
partisan_scores_2020.agg(["mean", "std", "min", "max"]).T

The fixture contains three presidential elections. Stacking them gives arrays with shape
`(plans, elections, districts)` for cross-election summaries.

In [ ]:
years = (2016, 2020, 2024)
dem_elections = np.stack(
    [tallies[f"pres_{year % 100:02d}_dem"].to_numpy() for year in years],
    axis=1,
)
rep_elections = np.stack(
    [tallies[f"pres_{year % 100:02d}_rep"].to_numpy() for year in years],
    axis=1,
)

wins_by_district = gs.formulas.party_wins_by_district(
    dem_elections,
    rep_elections,
)
pd.DataFrame(wins_by_district[:5], columns=districts)

In [ ]:
cross_election_scores = pd.DataFrame(
    {
        "competitive_contests": gs.formulas.competitive_contests(
            dem_elections,
            rep_elections,
            points_within=0.05,
        ),
        "swing_districts": gs.formulas.swing_districts(
            dem_elections,
            rep_elections,
        ),
        "democratic_districts": gs.formulas.party_districts(
            dem_elections,
            rep_elections,
        ),
        "republican_districts": gs.formulas.opposition_party_districts(
            dem_elections,
            rep_elections,
        ),
        "aggregate_democratic_seats": gs.formulas.aggregate_seats(
            dem_elections,
            rep_elections,
        ),
        "mean_signed_seat_vote_gap": gs.formulas.mean_signed_seat_vote_gap(
            dem_elections,
            rep_elections,
        ),
        "mean_absolute_seat_vote_gap": gs.formulas.mean_absolute_seat_vote_gap(
            dem_elections,
            rep_elections,
        ),
    }
)
cross_election_scores.agg(["mean", "std", "min", "max"]).T

## Derive population, demographic, and compactness scores

Population and demographic functions use district tallies from the same streamed table.
Schwartzberg compactness is derived from the scoring-engine Polsby-Popper output.

In [ ]:
population = tallies["total_pop_20"].to_numpy()
voting_age_population = tallies["total_vap_20"].to_numpy()
black_voting_age_population = tallies["bvap_20"].to_numpy()

population_deviation = gs.formulas.population_deviations(population)
bvap_share = gs.formulas.demographic_shares(
    black_voting_age_population,
    voting_age_population,
)
population_scores = pd.DataFrame(
    {
        "max_absolute_deviation": gs.formulas.max_absolute_population_deviation(
            population,
            relative=True,
        ),
        "maximum_deviation": gs.formulas.max_population_deviation(
            population,
            relative=True,
        ),
        "districts_above_40_percent_BVAP": gs.formulas.districts_above_threshold(
            black_voting_age_population,
            voting_age_population,
            threshold=0.4,
        ),
    }
)
population_scores.agg(["mean", "std", "min", "max"]).T

In [ ]:
polsby = run.read("polsby_popper_graph", expand_repetitions=True).to_numpy()
schwartzberg = gs.formulas.schwartzberg(polsby)

pd.concat(
    {
        "population deviation": pd.DataFrame(
            population_deviation[:5],
            columns=districts,
        ),
        "BVAP share": pd.DataFrame(
            bvap_share[:5],
            columns=districts,
        ),
        "Schwartzberg": pd.DataFrame(
            schwartzberg[:5],
            columns=districts,
        ),
    },
    axis="columns",
)

## Inspect Eguia and region-by-district tallies

`Eguia` is a first-class evaluator metric, so callers name its vote, region, and population
columns once rather than manually assembling a regional benchmark. `TallyByRegion` serves the
different task of exposing region-by-district values. For one plan, regions form the row index;
the first column level contains meaningful value names, and the second contains ordered district
labels.

In [ ]:
county_totals = single["county_totals"]
display(county_totals.head())

run.read("eguia_2020", expand_repetitions=True).describe()

## Result contracts

`evaluate` returns `PlanEvalResult`; indexing it by metric name returns a scalar, Series, or
DataFrame with semantic labels. `evaluate_many` returns `EnsembleEvalResult`; its first axis uses
`sample_ids` when supplied. Region results use `(sample, region)` rows for ensembles and region
rows for one plan, with `(metric, district)` columns in both cases. `array(name)` is available
when a canonical immutable NumPy view is preferable.

`evaluate_stream` returns `EvaluationRun` after atomically writing one Parquet table per metric plus
`manifest.json`. `frames` exposes accepted-frame offsets and repetition counts. `read()` restores
the same logical pandas shapes at either accepted-frame or expanded-sample resolution, while
`raw()` remains available for physical Parquet access. Large results can instead be processed
with `iter_batches()`, `iter_raw_batches()`, or `iter_frame_batches()` without materializing a
whole table. Array formulas deliberately remain
separate, so they work with in-memory results, streamed tables, or arrays produced elsewhere.

Sequence assignments must follow graph-node order. Mapping assignments are aligned by node
identifier; both forms are captured by the exported `gerrytools.scoring.Assignment` alias.
Geometry must be projected for area and distance metrics. Consult each metric or
formula docstring for its formula, tie convention, turnout model, and literature
references.